# Ling 3.0 Tiny: LoRA + agentic GSPO on AppWorld
This notebook runs a deliberately tiny, real before/after experiment. AppWorld database-state evaluators supply rewards; held-out `dev` task IDs are never used for training. The smoke result validates the pipeline, not the hypothesis.

## 1. Runtime and GPU check

In [12]:
import json
import os
import platform
import subprocess
import sys
import shutil
from pathlib import Path

import torch

assert platform.system() == "Linux", "Use a Google Colab Linux runtime."
assert torch.cuda.is_available(), "Select an NVIDIA GPU runtime."

props = torch.cuda.get_device_properties(0)

runtime = {
    "python": sys.executable,
    "gpu": props.name,
    "vram_gib": round(props.total_memory / 2**30, 1),
    "cuda": torch.version.cuda,
    "torch": torch.__version__,
}

print(json.dumps(runtime, indent=2))

assert runtime["vram_gib"] >= 30, "Use an A100 40GB or larger."

{
  "python": "/usr/bin/python3",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "vram_gib": 39.5,
  "cuda": "12.8",
  "torch": "2.11.0+cu128"
}


## 2. Clone and install this repository
Set `REPO_URL` to the GitHub repository containing this notebook. A private repository requires a suitable Git credential or token in the URL. No model token is printed.

In [ ]:
REPO_URL = "https://github.com/JoshuaEworo/agent-rl-poc.git"

PROJECT = Path("/content/agent-rl-poc")
PROJECT_SRC = PROJECT / "src"

if PROJECT.exists():
    shutil.rmtree(PROJECT)

subprocess.run(
    ["git", "clone", REPO_URL, str(PROJECT)],
    check=True,
)

assert PROJECT_SRC.exists(), f"Missing {PROJECT_SRC}"

os.chdir(PROJECT)

# Make our src-layout package visible immediately.
sys.path.insert(0, str(PROJECT_SRC))

existing = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(PROJECT_SRC)
    if not existing
    else f"{PROJECT_SRC}:{existing}"
)

# Install only project/dev dependencies.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    check=True,
)

print("Project:", PROJECT)
print("Project src:", PROJECT_SRC)

## 3. Install AReno and AppWorld
AReno builds its CUDA extension from the audited source revision. If Colab explicitly requests a restart after installation, restart once and continue at section 4.

In [13]:
import re

ARENO_REV = "48d07c54051c41bf36218f99bce1c3697e9ba63c"
ARENO_SRC = Path("/content/AReno")

if ARENO_SRC.exists():
    shutil.rmtree(ARENO_SRC)

subprocess.run(
    ["git", "clone", "https://github.com/inclusionAI/AReno.git", str(ARENO_SRC)],
    check=True,
)

subprocess.run(
    ["git", "-C", str(ARENO_SRC), "checkout", ARENO_REV],
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-U",
        "pip", "setuptools", "wheel", "packaging", "psutil", "ninja"
    ],
    check=True,
)

cuda = torch.version.cuda
assert cuda is not None

torch_match = re.match(r"(\d+\.\d+)", torch.__version__)
assert torch_match

torch_mm = torch_match.group(1)
cuda_index = "cu" + cuda.replace(".", "")
astral_index = f"https://wheels.astral.sh/simple/{cuda_index}/"

flash_version = f"2.8.3.post1+cu.{cuda}.torch.{torch_mm}"

print("PyTorch:", torch.__version__)
print("CUDA:", cuda)
print("Trying FlashAttention:", flash_version)

env = os.environ.copy()
env["MAX_JOBS"] = "4"

try:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-deps",
            f"flash-attn=={flash_version}",
            "--extra-index-url", astral_index,
        ],
        check=True,
        env=env,
    )

    subprocess.run(
        [
            sys.executable,
            "-c",
            "import flash_attn; print('FlashAttention:', flash_attn.__version__)"
        ],
        check=True,
    )

except subprocess.CalledProcessError:
    print("Prebuilt FlashAttention unavailable; building from source.")

    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "flash-attn"],
        check=False,
    )

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "flash-attn>=2.7",
            "--no-build-isolation",
            "--no-cache-dir",
        ],
        check=True,
        env=env,
    )

try:
    subprocess.run(
        ["bash", str(ARENO_SRC / "scripts/install.sh")],
        cwd=ARENO_SRC,
        env=env,
        check=True,
    )
except subprocess.CalledProcessError:
    log_path = Path.home() / ".local/state/areno/install.log"

    if log_path.exists():
        print("\n====== AReno install log ======\n")
        print("\n".join(log_path.read_text(errors="replace").splitlines()[-200:]))

    raise

# Make the checked-out AReno source importable from this notebook
sys.path.insert(0, str(ARENO_SRC))

old_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = f"{ARENO_SRC}:{old_pythonpath}"

print("areno executable:", shutil.which("areno"))

PyTorch: 2.11.0+cu128
CUDA: 12.8
Trying FlashAttention: 2.8.3.post1+cu.12.8.torch.2.11
areno executable: /usr/local/bin/areno


## 4. Validate imports, credentials, and experiment split

In [14]:
APPWORLD_REV = "42b5bcf3cd334fee33f0c37c02070a9f5807add5"
APPWORLD_SRC = Path("/content/appworld")

subprocess.run(["apt-get", "update"], check=True)
subprocess.run(["apt-get", "install", "-y", "git-lfs"], check=True)
subprocess.run(["git", "lfs", "install"], check=True)

if APPWORLD_SRC.exists():
    shutil.rmtree(APPWORLD_SRC)

subprocess.run(
    [
        "git", "clone",
        "https://github.com/StonyBrookNLP/appworld.git",
        str(APPWORLD_SRC),
    ],
    check=True,
)

subprocess.run(
    ["git", "-C", str(APPWORLD_SRC), "checkout", APPWORLD_REV],
    check=True,
)

subprocess.run(
    ["git", "-C", str(APPWORLD_SRC), "lfs", "pull"],
    check=True,
)

bundle = APPWORLD_SRC / "src/appworld/.source/apps.bundle"

assert bundle.exists()
print("apps.bundle size:", bundle.stat().st_size, "bytes")

with bundle.open("rb") as f:
    assert b"git-lfs.github.com/spec" not in f.read(100), \
        "AppWorld LFS data was not downloaded."

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--upgrade", str(APPWORLD_SRC)
    ],
    check=True,
)

def run_verbose(cmd, cwd=None):
    print(f"\n>>> {' '.join(cmd)}\n")

    result = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(cmd)}"
        )

# Unpack AppWorld apps/tests
run_verbose(["appworld", "install"])

# Avoid AppWorld's site-packages/dist-packages detection issue.
from appworld import update_root
from appworld.download import download_data

os.chdir(PROJECT)
update_root(str(PROJECT))

print("\n>>> Downloading AppWorld data\n")
download_data(mode="minimal")

assert (PROJECT / "data").exists()

print("\n✅ AppWorld ready at:", PROJECT / "data")

apps.bundle size: 193950 bytes

>>> appworld install



Output()

Unpacked apps source code 
  - from: /usr/local/lib/python3.13/dist-packages/appworld/.source/apps.bundle
  - in  : /usr/local/lib/python3.13/dist-packages/appworld
Unpacked tests source code 
  - from: /usr/local/lib/python3.13/dist-packages/appworld/.source/tests.bundle
  - in  : /root/.cache/appworld/tests


>>> Downloading AppWorld data



📦 Unpacking bundle

🚀 Data prepared at /content/ling-agent-rl-poc/data


✅ AppWorld ready at: /content/ling-agent-rl-poc/data


## 5. Start baseline Ling and run one real trajectory

In [15]:
os.chdir(PROJECT)

# Ensure both repos are visible to notebook AND subprocess Python.
paths = [
    str(PROJECT_SRC),
    str(ARENO_SRC),
]

for path in reversed(paths):
    if path not in sys.path:
        sys.path.insert(0, path)

existing = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = ":".join(
    paths + ([existing] if existing else [])
)

# HF token, if required
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

import areno
import appworld
import ling_agent_rl

print("AReno:", areno.__file__)
print("AppWorld:", appworld.__file__)
print("Project package:", ling_agent_rl.__file__)

assert str(PROJECT_SRC) in ling_agent_rl.__file__

# Point AppWorld at the correct project root again
from appworld import update_root
update_root(str(PROJECT))

# Run repo tests
subprocess.run(
    [sys.executable, "-m", "pytest"],
    cwd=PROJECT,
    env=os.environ.copy(),
    check=True,
)

from ling_agent_rl.config import load_config
from ling_agent_rl.train import prepare_dataset

CONFIG_PATH = "configs/smoke.yaml"

config = load_config(CONFIG_PATH)

dataset_path, train_ids, eval_ids = prepare_dataset(config)

assert set(train_ids).isdisjoint(eval_ids)

print({
    "dataset": str(dataset_path),
    "train_tasks": train_ids,
    "held_out_tasks": eval_ids,
})

print("\n✅ Validation complete")

AReno: /content/AReno/areno/__init__.py
AppWorld: /usr/local/lib/python3.13/dist-packages/appworld/__init__.py
Project package: /content/ling-agent-rl-poc/src/ling_agent_rl/__init__.py
{'dataset': '/content/ling-agent-rl-poc/artifacts/smoke/train_tasks.jsonl', 'train_tasks': ['ccb4494_2', 'e3d6c94_2'], 'held_out_tasks': ['68ee2c9_1', '0d8a4ee_3']}

✅ Validation complete


## 6. Start vanilla Ling

In [16]:
import time
import urllib.request

ARENO_BIN = shutil.which("areno")
assert ARENO_BIN, "areno executable not found"

def start_server(adapter=None, port=8000):
    cmd = [
        ARENO_BIN,
        "serve",
        "--model-path", config.model,
        "--model-hub", "hf",
        "--world-size", "1",
        "--tp-size", "1",
        "--port", str(port),
    ]

    if adapter:
        cmd += ["--lora-adapter-path", str(adapter)]

    print("Starting:", " ".join(cmd))

    proc = subprocess.Popen(
        cmd,
        cwd=PROJECT,
        env=os.environ.copy(),
    )

    for _ in range(180):
        if proc.poll() is not None:
            raise RuntimeError(
                f"AReno server exited early with code {proc.returncode}"
            )

        try:
            urllib.request.urlopen(
                f"http://127.0.0.1:{port}/v1/models",
                timeout=2,
            )
            print("✅ AReno server ready")
            return proc
        except Exception:
            time.sleep(2)

    proc.terminate()
    raise TimeoutError("AReno server did not become ready")

baseline_server = start_server()

Starting: /usr/local/bin/areno serve --model-path inclusionAI/Ling-3.0-tiny --model-hub hf --world-size 1 --tp-size 1 --port 8000
✅ AReno server ready


## 7. One real native-LoRA GSPO update and adapter save
This invokes AReno's native Bailing-MoE V3 LoRA and agentic rollout hook. It does not silently fall back to full-parameter training.

In [17]:
import asyncio

from openai import AsyncOpenAI
from ling_agent_rl.agent import run_episode

async def one_real_episode():
    client = AsyncOpenAI(
        base_url="http://127.0.0.1:8000/v1",
        api_key="unused",
    )

    try:
        trajectory, *rest = await run_episode(
            client=client,
            task_id=eval_ids[0],
            model="policy",
            rollout_config=config.rollout,
            experiment_name="ling_rl_preflight",
        )

        return trajectory

    finally:
        await client.close()

preflight = await one_real_episode()

print(
    json.dumps(
        preflight.to_dict(),
        indent=2,
        default=str,
    )
)

assert preflight.tool_calls > 0, \
    "Ling never invoked AppWorld."

print("\n✅ Real multistep trajectory completed")

────────────────────────────────────────────────── Overall Stats ──────────────────────────────────────────────────

Num Passed Tests : 1

Num Failed Tests : 4

Num Total  Tests : 5

───────────────────────────────────────────────────── Passes ──────────────────────────────────────────────────────

...

>> Failed Requirement

assert the files have been renamed (with proper content) as per private_data.this_year_start_to_end_file_path.

```python
with test(
    """
    assert the files have been renamed (with proper content) as per private_data.this_year_start_to_end_file_path.
    """
):
    start_content = []
    end_content = []
    for start_path, end_path in private_data.this_year_start_to_end_file_path.items():
```
----------

{
  "task_id": "68ee2c9_1",
  "user_request": "In my file system, add the prefix \"YYYY-MM-DD_\" to all file names in the ~/downloads/ directory, based on their creation dates, and then move all files not from this year to ~/trash/.",
  "environment": {
    "task_id": "68ee2c9_1",
    "instruction": "In my file system, add the prefix \"YYYY-MM-DD_\" to all file names in the ~/downloads/ directory, based on their creation dates, and then move all files not from this year to ~/trash/.",
    "datetime": "2023-05-18T12:00:00",
    "allowed_apps": [
      "api_docs",
      "supervisor",
      "amazon",
      "phone",
      "file_system",
      "spotify",
      "venmo",
      "gmail",
      "splitwise",
      "simple_note",
      "todoist"
    ],
    "db_version": "0.2.0"
  },
  "model_config": {
    "model": "policy",
    "adapter": "endpoint-selected"
  },
  "steps": [],
  "reward": 0.0,
  "success": false,
  "terminal_reason": "timeout",
  "evaluation": {
    "success": false,
    "diffic

AssertionError: Ling never invoked AppWorld.

## 8. Baseline Eval

In [ ]:
from ling_agent_rl.evaluate import run_evaluation

baseline = run_evaluation(
    config,
    eval_ids,
    base_url="http://127.0.0.1:8000",
    label="baseline",
)

print(json.dumps(baseline, indent=2))

baseline_server.terminate()
baseline_server.wait(timeout=30)

print("\n✅ Baseline finished")

## 9. Actual LoRA + GSPO smoke update

In [ ]:
from ling_agent_rl.train import train, verify_adapter

os.chdir(PROJECT)

train(config)

adapter = verify_adapter(
    Path(config.artifact_dir) / "adapter"
)

print("✅ Validated LoRA adapter:", adapter)

## 10. Reload LoRA and eval

In [ ]:
trained_server = start_server(adapter=adapter)

trained = run_evaluation(
    config,
    eval_ids,
    base_url="http://127.0.0.1:8000",
    label="trained",
)

trained_server.terminate()
trained_server.wait(timeout=30)

print(json.dumps(trained, indent=2))

print("\n✅ Trained evaluation finished")

## 11. Compare

In [ ]:
from ling_agent_rl.evaluate import compare

comparison = compare(baseline, trained)

comparison_path = (
    Path(config.artifact_dir) / "comparison.json"
)

comparison_path.write_text(
    json.dumps(comparison, indent=2)
)

print(json.dumps(comparison, indent=2))

print("\nSaved:", comparison_path)
print(
    "\nRemember: the smoke run checks that RL works end-to-end. "
    "Two held-out tasks are not evidence of general improvement."
)

## 12. Optional save to Drive

In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

    destination = (
        Path("/content/drive/MyDrive/ling-agent-rl-artifacts")
        / Path(config.artifact_dir).name
    )

    shutil.copytree(
        config.artifact_dir,
        destination,
        dirs_exist_ok=True,
    )

    print("Saved to:", destination)